# Exercise: Prophet as an Auto-Tuned Classical Model

Prophet automates trend and seasonality detection. Fit a Prophet model with temperature as a regressor, inspect its components, and compare its forecast to the ARIMA models from earlier.

Prophet decomposes the signal into trend + yearly seasonality + regressor effects. Its intervals are simulation-based rather than analytic, so they may differ in width from SARIMAX even when point forecasts agree.

Implement the three tasks marked with `# YOUR CODE HERE`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
HORIZON = 12


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date").asfreq("MS")
train_df = df.loc[:"2023-12-01"]
train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]
seasonal_avg_temp = train_df.groupby(train_df.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_avg_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS"),
)
# ARIMA baseline for comparison
arima_fit = SARIMAX(train_demand, order=(2, 1, 1)).fit(disp=False)
arima_fc = arima_fit.get_forecast(HORIZON)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int()
print("Data and ARIMA baseline ready.")


def fit_prophet(train_df):
    """
    Fit a Prophet model with yearly seasonality and temperature regressor.
    
    Parameters
    ----------
    train_df : pd.DataFrame
        Must have columns ds (datetime), y (target), and avg_temp_f.
    
    Returns
    -------
    prophet.Prophet
        Fitted Prophet model.
    """
    m = Prophet(yearly_seasonality=..., weekly_seasonality=..., daily_seasonality=...)
    m.add_regressor(...)
    m.fit(train_df[["ds", "y", ...]])
    return m

prophet_model = fit_prophet(train_df)

In [ ]:
def prophet_forecast(model, train_df, future_temp, horizon):
    """
    Generate Prophet forecast with temperature regressor.
    
    Parameters
    ----------
    model : prophet.Prophet
    train_df : pd.DataFrame
    future_temp : pd.Series
    horizon : int
    
    Returns
    -------
    pd.DataFrame
        Forecast with yhat, yhat_lower, yhat_upper.
    """
    future = model.make_future_dataframe(periods=..., freq=...)
    future["avg_temp_f"] = list(train_df["avg_temp_f"].values) + list(future_temp.values)
    forecast = model.predict(future)
    return forecast.iloc[-horizon:]

prophet_pred = prophet_forecast(prophet_model, train_df, future_temp, HORIZON)

future_full = prophet_model.make_future_dataframe(periods=HORIZON, freq="MS")
future_full["avg_temp_f"] = list(train_df["avg_temp_f"].values) + list(future_temp.values)
forecast_full = prophet_model.predict(future_full)

fig = prophet_model.plot_components(forecast_full)
for ax in fig.axes:
    ax.tick_params(axis="both", labelsize=14)
fig.suptitle("Prophet Components", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_demand.index, train_demand.values, color=UN["Black"], label="Historical")
ax.plot(arima_mean.index, arima_mean.values, color=UB["Brand Blue"], linestyle="--", label="ARIMA")
ax.fill_between(arima_mean.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1], color=UB["Brand Blue"], alpha=0.1)
ax.plot(prophet_pred.index, prophet_pred["yhat"].values, color=US["Green"], linestyle="--", label="Prophet")
ax.fill_between(prophet_pred.index, prophet_pred["yhat_lower"].values, prophet_pred["yhat_upper"].values, color=US["Green"], alpha=0.1)
ax.set_ylabel("Demand (MWh)", fontsize=16)
ax.set_title("ARIMA vs Prophet Forecast", fontsize=18, fontweight="bold")
ax.tick_params(axis="both", labelsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## Task 3: Plot Prophet Components

Use `prophet_model.plot_components()` to visualize the trend and seasonality decomposition.

**Hint:** You need the full forecast DataFrame (training + future periods) for `plot_components`.

In [ ]:
future_full = prophet_model.make_future_dataframe(periods=HORIZON, freq="MS")
future_full["avg_temp_f"] = list(train_df["avg_temp_f"].values) + list(future_temp.values)
forecast_full = prophet_model.predict(future_full)

fig = prophet_model.plot_components(forecast_full)
for ax in fig.axes:
    ax.tick_params(axis="both", labelsize=14)
fig.suptitle("Prophet Components", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_demand.index, train_demand.values, color=UN["Black"], label="Historical")
ax.plot(arima_mean.index, arima_mean.values, color=UB["Brand Blue"], linestyle="--", label="ARIMA")
ax.fill_between(arima_mean.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1], color=UB["Brand Blue"], alpha=0.1)
ax.plot(prophet_pred.index, prophet_pred["yhat"].values, color=US["Green"], linestyle="--", label="Prophet")
ax.fill_between(prophet_pred.index, prophet_pred["yhat_lower"], prophet_pred["yhat_upper"], color=US["Green"], alpha=0.1)
ax.set_ylabel("Demand (MWh)")
ax.legend()
plt.tight_layout()
ax.set_title("Plot", fontsize=18, fontweight="bold")
ax.set_ylabel("Value", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
plt.show()
